# DermaMap preprocessing — visual debugging

Side-by-side view of every step in the 8-stage pipeline.

**Setup:**
```bash
pip install opencv-python numpy matplotlib scikit-image
```

Place real mole photos in `test_images/` (project root). The notebook will
pick them up automatically. Synthetic samples are generated as a fallback.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'model' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import cv2
import numpy as np
import matplotlib.pyplot as plt

from model.preprocessing import (
    debug_pipeline,
    preprocess_for_inference,
    PipelineTimings,
)

## 1. Load test images
Drop a few `.jpg` / `.png` files into `test_images/` at the project root.
If the directory is empty, synthetic samples are used.

In [ ]:
TEST_DIR = ROOT / 'test_images'

def load_real():
    if not TEST_DIR.exists():
        return []
    files = sorted(p for p in TEST_DIR.iterdir()
                   if p.suffix.lower() in {'.jpg', '.jpeg', '.png'})
    return [
        (p.name, cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB))
        for p in files
    ][:10]

def synthetic(seed: int):
    rng = np.random.RandomState(seed)
    img = np.full((480, 480, 3), (210, 175, 140), dtype=np.uint8)
    cx, cy = rng.randint(180, 300, size=2)
    r       = rng.randint(50, 110)
    colour  = (rng.randint(40, 100), rng.randint(30, 70), rng.randint(30, 70))
    cv2.circle(img, (cx, cy), r, colour, thickness=-1)
    noise = rng.normal(0, 10, img.shape).astype(np.int16)
    img = np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    return f'synthetic_{seed}', img

samples = load_real()
if not samples:
    print('No real images in test_images/, using 6 synthetic samples')
    samples = [synthetic(i) for i in range(6)]
print(f'Loaded {len(samples)} images')

## 2. Visualise every stage for the first sample

In [ ]:
def show_pipeline(name: str, img: np.ndarray):
    stages = debug_pipeline(img, enable_quality_check=False)
    titles = [
        ('raw',             stages.raw),
        ('1. mask',         stages.mask),
        ('2. cropped',      stages.cropped),
        ('3. hair removed', stages.hair_removed),
        ('4. shades of gray', stages.color_corrected),
        ('5. CLAHE',        stages.clahe),
        ('6. resized 224',  stages.resized),
        ('7. tensor (clipped to uint8)', stages.tensor.astype(np.uint8) if stages.tensor is not None else None),
    ]
    fig, axes = plt.subplots(2, 4, figsize=(18, 9))
    for ax, (title, im) in zip(axes.flat, titles):
        if im is None:
            ax.set_title(f'{title}\n(skipped)')
            ax.axis('off'); continue
        if im.ndim == 2:
            ax.imshow(im, cmap='gray')
        else:
            ax.imshow(im)
        ax.set_title(title)
        ax.axis('off')
    fig.suptitle(f'{name} — quality: {stages.quality}', y=1.02)
    plt.tight_layout()
    plt.show()

show_pipeline(*samples[0])

## 3. All samples — quality check report

In [ ]:
rows = []
for name, img in samples:
    t = PipelineTimings()
    tensor, q = preprocess_for_inference(img, enable_quality_check=False, timings=t)
    rows.append({
        'name':       name,
        'ok':         q['ok'],
        'reason':     q['reason'],
        'sharpness':  round(q['metrics']['sharpness'], 1),
        'brightness': round(q['metrics']['brightness'], 1),
        'contrast':   round(q['metrics']['contrast'], 1),
        'coverage':   round(q['metrics']['lesion_coverage'], 3),
        'total_ms':   round(t.total, 1),
    })

try:
    import pandas as pd
    df = pd.DataFrame(rows)
    display(df)
except ImportError:
    for r in rows: print(r)

## 4. Per-step timing breakdown

In [ ]:
name, img = samples[0]
t = PipelineTimings()
preprocess_for_inference(img, enable_quality_check=False, timings=t)
labels = ['quality', 'segment', 'crop', 'hair', 'SoG', 'CLAHE', 'resize', 'float']
values = [t.quality_check, t.segment_lesion, t.crop_to_lesion, t.hair_removal,
          t.color_constancy, t.clahe, t.resize, t.to_float]
plt.figure(figsize=(10, 4))
plt.bar(labels, values)
plt.ylabel('ms')
plt.title(f'Per-step time on {name} ({t.total:.1f} ms total)')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 5. Sanity: verify the final tensor is in [0, 255], not [0, 1]

In [ ]:
tensor, _ = preprocess_for_inference(samples[0][1], enable_quality_check=False)
print('dtype:', tensor.dtype)
print('shape:', tensor.shape)
print('min/mean/max:', tensor.min(), tensor.mean().round(1), tensor.max())
assert tensor.max() > 1.0, '⚠ tensor looks /255-normalised — model will see garbage!'
print('OK — input range looks correct.')